In [128]:
import models.model as modelgen
from configs import config
import torch
import torch.nn as nn
import utils.log as log
import json
import numpy as np

In [136]:
def get_dct_matrix(N):
    dct_m = np.eye(N)
    for k in np.arange(N):
        for i in np.arange(N):
            w = np.sqrt(2 / N)
            if k == 0:
                w = np.sqrt(1 / N)
            dct_m[k, i] = w * np.cos(np.pi * (i + 1 / 2) * k / N)
    idct_m = np.linalg.inv(dct_m)
    return dct_m, idct_m

dct_m, idct_m = get_dct_matrix(4)

In [167]:
a = torch.arange(32*32*99*3, dtype=torch.double) - 5
B = a.reshape((32, 32, 99*3))
dct_m, idct_m = get_dct_matrix(32)
dct_m = torch.tensor(dct_m).unsqueeze(0)
idct_m = torch.tensor(idct_m).unsqueeze(0)
torch.matmul(dct_m, B).shape

torch.Size([32, 32, 297])

In [165]:
torch.matmul(idct_m, torch.matmul(dct_m, B)).shape

torch.Size([32, 32, 297])

In [166]:
torch.matmul(idct_m, torch.matmul(dct_m, B)).reshape(32,16,297)

RuntimeError: shape '[32, 16, 297]' is invalid for input of size 304128

In [113]:
from torch import linalg as LA

a = torch.arange(32*32*99*3, dtype=torch.float) - 5
B = a.reshape((32, 32, 99*3))
c = torch.arange(32*32*99*3, dtype=torch.float)
D = c.reshape((32, 32, 99*3))
loss = torch.mean(LA.vector_norm(B-D, 2, -1))
LA.vector_norm(B-D, 2, -1).shape
loss

tensor(86.1684)

In [114]:
a = torch.arange(32*32*99*3, dtype=torch.float) - 5
B = a.reshape((-1, 99*3))
c = torch.arange(32*32*99*3, dtype=torch.float)
D = c.reshape((-1, 99*3))
loss = torch.mean(LA.vector_norm(B-D, 2, 1))
LA.vector_norm(B-D, 2, 1).shape
loss

tensor(86.1684)

In [115]:
def gen_velocity(m):
    dm = m[:, 1:] - m[:, :-1]
    return dm
dloss = LA.vector_norm(gen_velocity(B)-gen_velocity(D), 2, -1)
dloss.shape

torch.Size([1024])

In [2]:
cfgs = config.Configurations('./configs/LSFB/siMLPe/vanilla.yaml')

logger = log.make_logger('./logs/testing/', "test", None)
logger.info("Run name : {run_name}".format(run_name='test'))
for k, v in cfgs.super_cfgs.items():
    logger.info("cfgs." + k + " =")
    logger.info(json.dumps(vars(v), indent=2))

[INFO] 2024-04-24 23:21:19 > Run name : test
[INFO] 2024-04-24 23:21:19 > cfgs.DATA =
[INFO] 2024-04-24 23:21:19 > {
  "name": "LSFB",
  "input_size": [
    32,
    99,
    3
  ],
  "num_classes": 610,
  "min_samples": 20,
  "max_len": 32,
  "oversample": true,
  "poses": [
    [
      "pose",
      "all"
    ],
    [
      "right_hand",
      "all"
    ],
    [
      "left_hand",
      "all"
    ],
    [
      "face",
      [
        61,
        39,
        0,
        269,
        291,
        405,
        17,
        181,
        33,
        159,
        133,
        145,
        46,
        52,
        65,
        55,
        263,
        386,
        362,
        374,
        276,
        282,
        295,
        285
      ]
    ]
  ],
  "num_keypoints": 99,
  "flip_p": 0.0,
  "scale": 0.0,
  "random_crop": false,
  "drop_frame": 0.0,
  "drop_keypoint": 0.0,
  "block_size": 5,
  "rot": 0.0
}
[INFO] 2024-04-24 23:21:19 > cfgs.MODEL =
[INFO] 2024-04-24 23:21:19 > {
  "backbone": "si

In [3]:
rank = torch.cuda.current_device()
model = modelgen.load_model(DATA=cfgs.DATA,
                                MODEL=cfgs.MODEL,
                                MODULES=cfgs.MODULES,
                                RUN=cfgs.RUN,
                                device=rank,
                                logger=logger)

[INFO] 2024-04-24 23:21:28 > Build the model.
[INFO] 2024-04-24 23:21:28 > Modules are located on './src/models.siMLPe'.
[INFO] 2024-04-24 23:21:28 > Number of parameters: 256212
[INFO] 2024-04-24 23:21:28 > Model(
  (arr0): Rearrange('b n d -> b d n')
  (arr1): Rearrange('b d n -> b n d')
  (motion_mlp): TransMLP(
    (mlps): Sequential(
      (0): MLPblock(
        (fc0): Temporal_FC(
          (fc): Linear(in_features=32, out_features=32, bias=True)
        )
        (norm0): SLayerNorm()
      )
      (1): MLPblock(
        (fc0): Temporal_FC(
          (fc): Linear(in_features=32, out_features=32, bias=True)
        )
        (norm0): SLayerNorm()
      )
      (2): MLPblock(
        (fc0): Temporal_FC(
          (fc): Linear(in_features=32, out_features=32, bias=True)
        )
        (norm0): SLayerNorm()
      )
      (3): MLPblock(
        (fc0): Temporal_FC(
          (fc): Linear(in_features=32, out_features=32, bias=True)
        )
        (norm0): SLayerNorm()
      )
   

In [4]:
from torchinfo import summary

batch_size = 32
summary(model, input_size=(batch_size, 32,99*3), depth=1)

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [32, 32, 297]             --
├─Linear: 1-1                            [32, 32, 297]             88,506
├─Rearrange: 1-2                         [32, 297, 32]             --
├─TransMLP: 1-3                          [32, 297, 32]             79,200
├─Rearrange: 1-4                         [32, 32, 297]             --
├─Linear: 1-5                            [32, 32, 297]             88,506
Total params: 256,212
Trainable params: 256,212
Non-trainable params: 0
Total mult-adds (M): 7.29
Input size (MB): 1.22
Forward/backward pass size (MB): 238.44
Params size (MB): 1.02
Estimated Total Size (MB): 240.68

In [5]:
# Generate a random input tensor
input_tensor = torch.randn(32, cfgs.DATA.input_size[0], cfgs.DATA.input_size[1]*cfgs.DATA.input_size[2]).to(rank)

# Pass the input tensor through the model
output_tensor = model(input_tensor)

# Print the output tensor
print(output_tensor.shape)

torch.Size([32, 32, 297])


In [6]:
from data.data_util import Dataset_, train_val_dataset, OversamplingWrapper
import utils.misc as misc

data_name = "LSFB"
data_dir = "/mnt/sda2/datasets/isolated-cont-sl/LSFB/"
poses = [["pose","all"],["right_hand","all"],["left_hand","all"], ["face",[61,39,0,269,291,405,17,181,33,159,133,145,46,52,65,55,263,386,362,374,276,282,295,285]]]
dset_used = 1
seed = 42
oversample = True

print("Load {name} train dataset.".format(name=data_name))
train_dataset = Dataset_(data_dir=data_dir,
                            train=True,
                            load_data_in_memory=False,
                            poses=poses,
                            max_len=32,
                            min_samples=20,
                            drop_frame=0.5,
                            drop_keypoint=0.5,
                            block_size=60,
                            flip_p=0.5,
                            scale=0.5,
                            rot=5)
print("Train dataset size: {dataset_size}".format(dataset_size=len(train_dataset)))


Load LSFB train dataset.
Train dataset size: 52350


In [7]:
import torch.utils.benchmark as benchmark

input = train_dataset[0]
values = torch.reshape(input[0], (1,input[0].shape[0],input[0].shape[1])).to(rank)
labels = torch.tensor([input[1]]).to(rank)

def stepmodel(input):
    with torch.autocast("cuda"):
        # Pass the input tensor through the model
        output_tensor = model(input)
    return output_tensor

t0 = benchmark.Timer(
    stmt='stepmodel(x)',
    setup='from __main__ import stepmodel',
    globals={'x': values},
    num_threads=4)

print(t0.timeit(100))

stepmodel(x)
setup: from __main__ import stepmodel
  13.62 ms
  1 measurement, 100 runs , 4 threads
